In [ ]:
import pandas as pd
import os
import json
import re
import ast
import random
from collections import defaultdict, deque, OrderedDict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
!unzip -q /content/'Raw JSON Argument Maps (IAT)-20250513T193232Z-1-001.zip'

In [ ]:
!rm -rf '/content/Raw JSON Argument Maps (IAT)/IATConspiracyReddit1'
!rm -rf '/content/Raw JSON Argument Maps (IAT)/PolarIs1'
!rm -rf '/content/Raw JSON Argument Maps (IAT)/US2016tv'
!rm -rf '/content/Raw JSON Argument Maps (IAT)/mm2012'


In [ ]:
!unzip -q /content/repurposed_maps.zip

In [ ]:
def count_tokens(text):
    return len(tokenizer.encode(text))

def build_i_to_l_mapping(nodes, edges):
    """
    This function develops the mapping between the proposition's inference node and its locution node.

    :param nodes:
    :param edges:
    :return:
    """
    id_to_node = {node["nodeID"]: node for node in nodes}
    child_map = defaultdict(list)
    parent_map = defaultdict(list)

    for edge in edges:
        child_map[edge["fromID"]].append(edge["toID"])
        parent_map[edge["toID"]].append(edge["fromID"])

    i_to_l_map = {}
    # for each "I" node parse its parent lines to find its "L"
    for node in nodes:
        if node["type"] == "I":
            visited = set()
            queue = deque([(node["nodeID"], [])])
            while queue:
                current, path = queue.popleft()
                visited.add(current)
                for parent in parent_map[current]:
                    if parent in visited:
                        continue
                    if id_to_node[parent]["type"] == "L":
                        i_to_l_map[node["nodeID"]] = parent
                        queue.clear()
                        break
                    queue.append((parent, path + [parent]))
    return i_to_l_map


from difflib import SequenceMatcher

def normalize_with_map(original: str):
    """
    Normalize text to lowercase alnum+space only, and return:
    - norm: normalized string
    - norm_to_orig: list mapping each norm char index -> original char index
    """
    norm_chars = []
    norm_to_orig = []
    for i, ch in enumerate(original):
        ch_low = ch.lower()
        if ch_low.isalnum() or ch_low.isspace():
            # collapse whitespace later, but keep mapping now
            norm_chars.append(ch_low if ch_low.isalnum() else " ")
            norm_to_orig.append(i)

    norm = "".join(norm_chars)
    # collapse spaces while maintaining a mapping
    collapsed = []
    collapsed_map = []
    prev_space = False
    for j, ch in enumerate(norm):
        if ch == " ":
            if prev_space:
                continue
            prev_space = True
            collapsed.append(" ")
            collapsed_map.append(norm_to_orig[j])
        else:
            prev_space = False
            collapsed.append(ch)
            collapsed_map.append(norm_to_orig[j])

    norm = "".join(collapsed).strip()
    # If stripped, adjust mapping accordingly
    # Find first/last non-space in collapsed
    if not norm:
        return "", []
    first = next((k for k, c in enumerate(collapsed) if c != " "), None)
    last = next((k for k in range(len(collapsed)-1, -1, -1) if collapsed[k] != " "), None)
    norm_to_orig_final = collapsed_map[first:last+1]
    return norm, norm_to_orig_final


def normalize_node(text: str) -> str:
    """
    Node normalization consistent with normalize_with_map:
    - remove speaker prefix
    - lowercase
    - remove non-alnum (keep spaces)
    - collapse whitespace
    """
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r"^[^:]{1,30}\s*:\s*", "", text)
    text = re.sub(r"\[deleted\]", "", text, flags=re.IGNORECASE)
    text = text.lower()

    # smart quotes/dashes normalize
    text = (text.replace("’", "'").replace("‘", "'")
                .replace("“", '"').replace("”", '"')
                .replace("–", "-").replace("—", "-"))

    # keep only alnum and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def best_span_match(node_norm: str, conv_norm: str, cursor_norm: int,
                    min_ratio: float = 0.62,
                    max_scan_tokens: int = 220,
                    len_fuzz: int = 3):
    """
    Find best matching span in conv_norm for node_norm starting at/after cursor_norm.
    Returns (start_norm, end_norm, ratio) or (None, None, 0.0)
    """
    if not node_norm:
        return None, None, 0.0

    # Fast path: exact substring
    exact = conv_norm.find(node_norm, cursor_norm)
    if exact != -1:
        return exact, exact + len(node_norm), 1.0

    # Tokenize conv and build token start indices (in conv_norm chars)
    # Also respect cursor by starting token search near it.
    conv_tokens = conv_norm.split()
    if not conv_tokens:
        return None, None, 0.0

    # Build token -> char start positions
    starts = []
    pos = 0
    for t in conv_tokens:
        # find next occurrence from pos (safe due to split reconstruction)
        idx = conv_norm.find(t, pos)
        starts.append(idx)
        pos = idx + len(t)

    node_tokens = node_norm.split()
    n = len(node_tokens)
    if n == 0:
        return None, None, 0.0

    min_len = max(2, n - len_fuzz)
    max_len = n + len_fuzz

    # Determine start token index based on cursor_norm
    start_tok = 0
    for i, st in enumerate(starts):
        if st >= cursor_norm:
            start_tok = i
            break

    best = (None, None, 0.0)

    # scan windows
    # limit scanning to avoid quadratic blowups
    scan_end = min(len(conv_tokens), start_tok + max_scan_tokens)

    for i in range(start_tok, scan_end):
        for L in range(min_len, max_len + 1):
            j = i + L
            if j > len(conv_tokens):
                break

            window_text = " ".join(conv_tokens[i:j])

            # quick prune: require at least 2 overlapping tokens
            overlap = len(set(node_tokens) & set(conv_tokens[i:j]))
            if overlap < 2 and n >= 4:
                continue

            ratio = SequenceMatcher(None, node_norm, window_text).ratio()
            if ratio > best[2]:
                # compute char span in conv_norm
                start_char = starts[i]
                end_char = starts[j - 1] + len(conv_tokens[j - 1])
                best = (start_char, end_char, ratio)

    if best[2] >= min_ratio:
        return best
    return None, None, best[2]

def process_argument_graph_recursive(data):
    nodes = data["nodes"]
    edges = data["edges"]
    id_to_node = {node["nodeID"]: node for node in nodes}

    relation_types = ("RA", "CA")

    def get_text(node_id): return id_to_node[node_id]["text"]
    def get_type(node_id): return id_to_node[node_id]["type"]
    def get_parents(nid): return [e["fromID"] for e in edges if e["toID"] == nid]
    def get_children(nid): return [e["toID"] for e in edges if e["fromID"] == nid]

    # Build I-to-L mapping
    inf_prop_mapping = build_i_to_l_mapping(nodes, edges)

    # Identify valid I nodes (must participate in RA/CA relation)
    valid_i_nodes = set()

    for edge in edges:
        from_type = get_type(edge["fromID"])
        to_type = get_type(edge["toID"])
        if from_type == "I" and to_type in relation_types:
            if edge["fromID"] not in valid_i_nodes:
                valid_i_nodes.add(edge["fromID"])
        elif to_type == "I" and from_type in relation_types:
            if edge["toID"] not in valid_i_nodes:
                valid_i_nodes.add(edge["toID"])

    # Select corresponding L nodes based on valid I-node involvement
    final_nodes = []
    added_l_ids = set()

    for i_id in valid_i_nodes:
        if i_id not in inf_prop_mapping:
            continue
        l_id = inf_prop_mapping[i_id]
        ya_ids = [pid for pid in get_parents(i_id) if get_type(pid) == "YA"]
        # the "I" node could have more than one "YA"

        # Select the YA whose parent is of type L for reasoning purposes
        selected_ya = None
        for ya_id in ya_ids:
            ya_parents = get_parents(ya_id)
            if any(get_type(p) == "L" for p in ya_parents):
                selected_ya = ya_id
                break

        # the first condition marks the node that provides the reasoning while the second condition enforces that the final node is unique and not processed before.
        if selected_ya and l_id not in added_l_ids:
            final_nodes.append({
                "reason": get_text(selected_ya),
                "node_id": l_id,
                "text": get_text(l_id)
            })
            added_l_ids.add(l_id)


    # Extract rels — only if RA/CA has exactly one I source and one I target
    final_rels = []
    for node in nodes:
        # only process nodes that are of relevant types.
        if node["type"] in relation_types:
            ra_id = node["nodeID"]
            i_sources = [pid for pid in get_parents(ra_id) if get_type(pid) == "I"]
            i_targets = [cid for cid in get_children(ra_id) if get_type(cid) == "I"]
            # an CA/RA node may more than one "I" node as source or target.
            for i_src in i_sources:
              for i_tgt in i_targets:
                if i_src in inf_prop_mapping and i_tgt in inf_prop_mapping:
                    # Find YA parents of RA/CA that further should have a L parent
                    ya_ids = [pid for pid in get_parents(ra_id) if get_type(pid) == "YA"]
                    selected_ya = None
                    for ya_id in ya_ids:
                        if any(get_type(p) == "TA" for p in get_parents(ya_id)):
                            selected_ya = ya_id
                            break
                    ya_text = get_text(selected_ya) if selected_ya else "Unknown"
                    # save its type
                    if node["type"] == "RA":
                      relation_type = "support"
                    elif node["type"] == "CA":
                      relation_type = "attack"
                    elif node["type"] == "MA":
                      relation_type = "rephrasing"
                    else:
                      raise ValueError(f"Unknown node type: {node['type']}")

                    final_rels.append({
                        "reason": ya_text,
                        "source_id": inf_prop_mapping[i_src],
                        "target_id": inf_prop_mapping[i_tgt],
                        "relation_type": relation_type
                    })

    return final_nodes, final_rels

def validate_and_remap(final_nodes, final_rels):
    """
    Remap node IDs to be sequential, but preserve original occurrence order.
    """
    # Preserve order of first occurrence
    seen = {}
    next_id = 1
    for n in final_nodes:
        old_id = n["node_id"]
        if old_id not in seen:
            seen[old_id] = str(next_id)
            next_id += 1

    remap = seen  # mapping from old_id to new sequential id

    # Remap nodes keeping original order
    remapped_nodes = [{
        "justification": n["reason"],
        "id": remap[n["node_id"]],
        "text": n["text"]
    } for n in final_nodes]

    # Remap rels, filter out those pointing to missing nodes
    remapped_rels = [{
        "justification": r["reason"],
        "source_id": remap[r["source_id"]],
        "target_id": remap[r["target_id"]],
        "relation_type": r["relation_type"]
    } for r in final_rels if r["source_id"] in remap and r["target_id"] in remap]

    # Find unused nodes
    used_ids = set(r["source_id"] for r in remapped_rels) | set(r["target_id"] for r in remapped_rels)
    unused_l_nodes = set(remap[k] for k in remap if remap[k] not in used_ids)

    return remapped_nodes, remapped_rels, unused_l_nodes

def explain_unused_l_nodes(unused_l_ids, nodes, edges):
    id_to_node = {node["nodeID"]: node for node in nodes}
    reasons = {}

    def get_type(nid): return id_to_node[nid]["type"]
    def get_text(nid): return id_to_node[nid]["text"]
    def get_children(nid): return [e["toID"] for e in edges if e["fromID"] == nid]
    def get_parents(nid): return [e["fromID"] for e in edges if e["toID"] == nid]
    for l_id in unused_l_ids:
        explanation = {"node_id": l_id, "text": get_text(l_id), "issues": []}
        ya_children = [cid for cid in get_children(l_id) if get_type(cid) == "YA"]
        if not ya_children:
            explanation["issues"].append("No YA child found")
            reasons[l_id] = explanation
            continue
        ya_id = ya_children[0]
        i_children = [cid for cid in get_children(ya_id) if get_type(cid) == "I"]
        if not i_children:
            explanation["issues"].append("YA exists but no I child found")
            reasons[l_id] = explanation
            continue
        i_id = i_children[0]
        ra_ca_children = [cid for cid in get_children(i_id) if get_type(cid) in ("RA", "MA", "CA")]
        if not ra_ca_children:
            explanation["issues"].append("I exists but does not connect to RA MA, or CA")
        else:
            explanation["issues"].append("I connects to RA/CA/MA, but the relation may be incomplete or unmatched")
        reasons[l_id] = explanation

    return reasons

def develop_argument_map_from_corpus(input_dir: str, output_dir: str):
    """

    :param input_dir:
    :param output_dir:
    :return: save the argument maps json values
    """

    folder_name = os.path.basename(input_dir)
    new_directory_path = os.path.join(output_dir, folder_name)

    if not os.path.exists(new_directory_path):
      os.makedirs(new_directory_path)
      print(f"New directory '{folder_name}' created at {new_directory_path}")
    else:
      print(f"Directory '{folder_name}' already exists.")

    stats = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            filepath = os.path.join(input_dir, filename)
            filename_base = os.path.splitext(filename)[0]
            with open(filepath, 'r') as f:
                data = json.load(f)

                raw_nodes, raw_rels = process_argument_graph_recursive(data)
                remapped_nodes, remapped_rels, unused_l_nodes = validate_and_remap(raw_nodes, raw_rels)

                final_outputs = {
                "nodes": remapped_nodes,
                "relations": remapped_rels
                 }

            with open(os.path.join(new_directory_path, filename), 'w') as out_f:
                    json.dump(final_outputs, out_f, indent=2)

def clean_text(text):
    if not text:
        return ""

    text = re.sub(r'\s+', ' ', text)
    text = text.lower()

    # normalize apostrophes
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # remove speaker tags
    text = re.sub(r"^[^:]{1,30} ?: ?", "", text, flags=re.MULTILINE)

    # remove punctuation (KEEP WORDS)
    text = re.sub(r"[^\w\s]", "", text)

    return text.strip()


def clean_node_text(text):
    if not text or not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r"^[^:]{1,30} ?: ?", "", text)
    text = re.sub(r"\[deleted\]", "", text, flags=re.IGNORECASE)

    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def find_node_position(node_text, conversation_text, cursor):
    if not node_text:
        return None, cursor

    pattern = re.escape(node_text.lower())
    match = re.search(pattern, conversation_text.lower()[cursor:])
    if match:
        pos = cursor + match.start()
        return pos, pos + len(node_text)

    return None, cursor


def develop_data_files(text_dir, json_dirs,
                       min_nodes=4, max_nodes=40,
                       min_relations=4, max_relations=40,
                       max_conv_size=1000,
                       min_conv_size=50,
                       max_context_examples_size=1200,
                       do_rephrasing: bool = False,
                       as_explained_structure: bool = True):

    folder_name = os.path.basename(text_dir)
    new_json_dir = os.path.join(json_dirs, folder_name)
    if not os.path.exists(new_json_dir):
        raise ValueError(f"The directory {new_json_dir} does not exist.")

    records = []
    total_dropped_units = 0
    total_conversations = 0
    total_conv_tokens = 0
    total_argument_units = 0
    total_supports = 0
    total_attacks = 0

    for filename in os.listdir(text_dir):
        if not filename.endswith(".json"):
            continue

        match = re.search(r"nodeset(\d+)\.json", filename)
        if not match:
            continue
        conversation_id = match.group(1)

        json_path = os.path.join(new_json_dir, filename)
        text_path = os.path.join(text_dir, filename.replace(".json", ".txt"))

        with open(json_path, "r") as f:
            graph = json.load(f)

        # check node conditions
        if not (min_nodes <= len(graph["nodes"]) < max_nodes):
            continue
        # check relation conditions
        if not (min_relations <= len(graph["relations"]) < max_relations):
            continue
        with open(text_path, 'r', encoding='utf-8') as f:
            conversation_text = f.read()

        # check length conditions
        if not (min_conv_size <= count_tokens(conversation_text) <= max_conv_size):
            continue

        # ---- Arrange nodes chronologically with REPAIR ----

        conv_original = conversation_text  # keep original (already read from file)
        conv_norm, norm_to_orig = normalize_with_map(conv_original)

        cursor_norm = 0
        valid_nodes = []
        dropped_nodes = []
        repaired_nodes = []

        for n in graph["nodes"]:
            old_id = n["id"]
            raw_text = n.get("text", "")

            node_norm = normalize_node(raw_text)

            start_norm, end_norm, score = best_span_match(
                node_norm=node_norm,
                conv_norm=conv_norm,
                cursor_norm=cursor_norm,
                min_ratio=0.62,       # tuneable
                max_scan_tokens=220,  # tuneable
                len_fuzz=3            # tuneable
            )

            if start_norm is None:
                dropped_nodes.append({"conversation_id": conversation_id, "node_id": old_id, "text": raw_text})
                continue

            # Map normalized span back to original conversation substring
            # norm_to_orig maps each norm char index -> original char index
            orig_start = norm_to_orig[start_norm] if start_norm < len(norm_to_orig) else 0
            orig_end = norm_to_orig[end_norm - 1] + 1 if (end_norm - 1) < len(norm_to_orig) else len(conv_original)

            conv_span = conv_original[orig_start:orig_end].strip()

            # If it wasn't an exact match (score < 1.0), we repaired it
            if score < 0.999:
                repaired_nodes.append({
                    "conversation_id": conversation_id,
                    "node_id": old_id,
                    "old_text": raw_text,
                    "new_text": conv_span,
                    "score": round(score, 3)
                })

            # Replace node text with conversational value
            n["text"] = conv_span
            n["__pos"] = orig_start  # chronological position in original conversation
            cursor_norm = end_norm   # advance in normalized space
            valid_nodes.append(n)

        # Report repairs and drops
        if repaired_nodes:
            print(f"\n[Repaired nodes] Conversation {conversation_id}")
            for r in repaired_nodes:
                print(f"  - Node {r['node_id']} (score={r['score']}): '{r['old_text']}' -> '{r['new_text']}'")

        if dropped_nodes:
            print(f"\n[Dropped nodes] Conversation {conversation_id}")
            for d in dropped_nodes:
                print(f"  - Node {d['node_id']}: {d['text']}")

        graph["nodes"] = valid_nodes

        # Enforce min_nodes after dropping
        if len(graph["nodes"]) < min_nodes:
            continue

        # Sort by chronological position
        graph["nodes"].sort(key=lambda x: x["__pos"])
        for node in graph["nodes"]:
            node.pop("__pos", None)


        old_to_new_id = {}
        for idx, node in enumerate(graph["nodes"]):
            old_to_new_id[node["id"]] = str(idx)
            node["id"] = str(idx)

            illocution = node.get("justification", "").strip()
            node["explanation"] = illocution if illocution else ""

        clean_relations = []
        for rel in graph["relations"]:
            s = rel["source_id"]
            t = rel["target_id"]
            if s in old_to_new_id and t in old_to_new_id:
                rel["source_id"] = old_to_new_id[s]
                rel["target_id"] = old_to_new_id[t]
                clean_relations.append(rel)

        graph["relations"] = clean_relations

        if len(graph["relations"]) < min_relations:
            continue

        # develop code for version 4
        if as_explained_structure:
            # explained = {
            #     "argument_units": [],
            #     "support_relationships": [],
            #     "attack_relationships": [],
            # }
            explained = {
                "argument_units": [],
                "relations": []
            }


            # develop the argument units
            for node in graph["nodes"]:
                explained["argument_units"].append({
                    "reason": node["explanation"],
                    "id": int(node["id"]),
                    "text": node["text"],
                })

            # develop the argument relations

            # for rel in graph["relations"]:
            #     src = int(rel["source_id"])
            #     tgt = int(rel["target_id"])
            #     reason = rel.get("reason", "")
            #     if rel["relation_type"] == "supporting":
            #         explained["support_relationships"].append({
            #             "reason": reason,
            #             "source_id": src,
            #             "target_id": tgt
            #         })
            #     elif rel["relation_type"] == "attacking":
            #         explained["attack_relationships"].append({
            #             "reason": reason,
            #             "source_id": src,
            #             "target_id": tgt
            #         })
            #     else:
            #         raise ValueError(f"Unknown relation type: {rel['relation_type']}")

            # argument_data = explained

            for rel in graph["relations"]:
                src = int(rel["source_id"])
                tgt = int(rel["target_id"])

                # Skip self-referencing relations (source_id == target_id) to prevent errors
                if src == tgt:
                    print(f"Warning: Skipping self-referencing relation (source_id == target_id) in conversation {conversation_id}. Relation: {rel}")
                    continue

                # ensure directionality in data points. Attack is always uni-directional with target always reported first in the text followed by its source. For support bidirectionality is valid, with also cases that target (claim) is reported before the source (premise). To simplify the setup, we maintain unidirectionality across the relations, and switch any target-source for support

                if rel["relation_type"] == "support" and src < tgt:
                    explained["relations"].append({
                        "source_id": tgt,
                        "target_id": src,
                        "relation_type": "support"
                    })

                elif rel["relation_type"] == "support" and src > tgt:
                    explained["relations"].append({
                        "source_id": src,
                        "target_id": tgt,
                        "relation_type": "support"
                    })
                elif rel["relation_type"] == "attack":
                    explained["relations"].append({
                        "source_id": src,
                        "target_id": tgt,
                        "relation_type": "attack"
                    })
                # Explicitly handle 'rephrasing' relations as they are valid types but not processed into 'relations' list.
                elif rel["relation_type"] == "rephrasing":
                    print(f"Warning: Skipping 'rephrasing' relation in conversation {conversation_id}. Relation: {rel}")
                    continue
                else:
                    raise ValueError(f"Unknown relation type: {rel['relation_type']}")

            argument_data = explained

        # develop data for version 3. (not important)
        else:
            if do_rephrasing:
                id_to_unit = {node["id"]: {
                    "id": node["id"],
                    "text": node["text"],
                    "explanation": node["explanation"],
                    "supports": [],
                    "attacks": [],
                    "rephrases": []
                } for node in graph["nodes"]}
            else:
                id_to_unit = {node["id"]: {
                    "id": node["id"],
                    "text": node["text"],
                    "explanation": node["explanation"],
                    "supports": [],
                    "attacks": []
                } for node in graph["nodes"]}

            for rel in graph["relations"]:
                src = rel["source_id"]
                tgt = rel["target_id"]
                if rel["relation_type"] == "supporting" and tgt not in id_to_unit[src]["supports"]:
                    id_to_unit[src]["supports"].append(tgt)
                elif rel["relation_type"] == "attacking" and tgt not in id_to_unit[src]["attacks"]:
                    id_to_unit[src]["attacks"].append(tgt)
                elif rel["relation_type"] == "rephrasing" and do_rephrasing:
                    if tgt not in id_to_unit[src]["rephrases"]:
                        id_to_unit[src]["rephrases"].append(tgt)

            argument_data = list(id_to_unit.values())

        # check for ambiguitites and fix any errors such as duplicate relations, nested arguments, etc.

        # Function to check if a unit is a subspan of another
        def is_subspan(smaller, larger):
            return smaller.strip() in larger.strip() and smaller.strip() != larger.strip()

        def clean_argument_structure(arg_obj):

          units = arg_obj["argument_units"]
          # supports = arg_obj["support_relationships"]
          # attacks = arg_obj["attack_relationships"]
          relations = arg_obj["relations"]

          id_text_map = {unit["id"]: unit["text"] for unit in units}
          unit_items = list(id_text_map.items())

          # Identify sub-span units (keep longer)
          subspan_ids = set()
          for i, (id_i, text_i) in enumerate(unit_items):
              for j, (id_j, text_j) in enumerate(unit_items):
                  if i == j:
                      continue
                  if is_subspan(text_i, text_j):
                      if len(text_i.strip()) < len(text_j.strip()):
                          subspan_ids.add(id_i)
                      else:
                          subspan_ids.add(id_j)

          cleaned_units = [u for u in units if u["id"] not in subspan_ids]

          # Fallback if too aggressive
          if len(cleaned_units) < 2:
              return arg_obj, 0

          drop_count = len(units) - len(cleaned_units)

          # Re-index IDs
          old_to_new_id = {u["id"]: i for i, u in enumerate(cleaned_units)}
          for i, u in enumerate(cleaned_units):
              u["id"] = i

          def update_and_filter_rels(rels):
              seen = set()
              updated = []
              for rel in rels:
                  s = rel["source_id"]
                  t = rel["target_id"]
                  if s in subspan_ids or t in subspan_ids:
                      continue
                  pair = tuple(sorted([s, t]))
                  if pair in seen:
                      continue
                  seen.add(pair)
                  updated.append({
                      "source_id": old_to_new_id[s],
                      "target_id": old_to_new_id[t],
                      "type": rel["relation_type"]
                  })
              return updated


          cleaned_relations = update_and_filter_rels(relations)
          # cleaned_supports = update_and_filter_rels(supports)
          # cleaned_attacks = update_and_filter_rels(attacks)

          return {
              "argument_units": cleaned_units,
              # "support_relationships": cleaned_supports,
              # "attack_relationships": cleaned_attacks
              "relations": cleaned_relations
          }, drop_count


        # clean up the single conversation record.
        # print(argument_data)
        cleaned_structure, dropped_count = clean_argument_structure(argument_data)



        total_dropped_units += dropped_count

        total_conversations += 1
        total_conv_tokens += count_tokens(conversation_text)

        if as_explained_structure:
            total_argument_units += len(cleaned_structure["argument_units"])

            total_supports += len([r for r in cleaned_structure["relations"] if r["type"] == "support"])
            total_attacks += len([r for r in cleaned_structure["relations"] if r["type"] == "attack"])

            # total_supports += len(cleaned_structure["support_relationships"])
            # total_attacks += len(cleaned_structure["attack_relationships"])
        else:
            total_argument_units += len(cleaned_structure)
            for unit in cleaned_structure:
                total_supports += len(unit.get("supports", []))
                total_attacks += len(unit.get("attacks", []))


        rel_counts = {"support": total_supports, "attack": total_attacks}
        if do_rephrasing:
            rel_counts["rephrasing"] = 0

        def assign_weight(rtype):
            return {"supporting": 1, "attacking": 5, "rephrasing": 0}.get(rtype, 0)

        record = {
            "conversation_id": conversation_id,
            "conversation_text": conversation_text,
            "argument_objects": cleaned_structure,
            "dropped_units": dropped_count,
            "relation_counts": rel_counts,
            "diversity_score": len([v for v in rel_counts.values() if v > 0]),
            "weighted_relations": sum(v * assign_weight(k) for k, v in rel_counts.items())
        }

        # count tokens for the record including the conv text and unit text.
        if as_explained_structure:
            record["token_count"] = count_tokens(conversation_text) + sum(
                count_tokens(n["text"]) for n in argument_data["argument_units"]
            )
        else:
            record["token_count"] = count_tokens(conversation_text) + sum(
                count_tokens(n["text"]) for n in argument_data
            )

        # save conversation and process next one
        records.append(record)


    # develop context examples
    # sorted first by having both support and attack, and weighed by prioritizing attack relation.
    sorted_contexts = sorted(
        [r for r in records if r["diversity_score"] >= 2],
        key=lambda x: (-x["diversity_score"], -x["weighted_relations"])
    )

    num_desired_contexts = 5
    context_examples = []
    total_tokens = 0
    seen_ids = set()

    # Pass 1: Try to select the most diverse/weighted examples that fit all token constraints
    for candidate in sorted_contexts:
        if len(context_examples) >= num_desired_contexts:
            break
        if candidate["conversation_id"] in seen_ids:
            continue
        if candidate["token_count"] > 0.4 * max_context_examples_size: # Skip if individual example is too large
            continue
        if total_tokens + candidate["token_count"] <= max_context_examples_size: # Skip if total size exceeded
            context_examples.append(candidate)
            total_tokens += candidate["token_count"]
            seen_ids.add(candidate["conversation_id"])

    # Pass 2: If not enough examples, try to fill remaining slots with smaller ones from remaining candidates,
    # relaxing the individual 0.4*max_context_examples_size constraint, but still respecting total_tokens constraint.
    if len(context_examples) < num_desired_contexts:
        # Get remaining candidates, sorted by token count (smallest first)
        remaining_candidates_by_size = sorted([c for c in sorted_contexts if c["conversation_id"] not in seen_ids],
                                              key=lambda x: x["token_count"])
        for candidate in remaining_candidates_by_size:
            if len(context_examples) >= num_desired_contexts:
                break
            # Now, individual example size (0.4 * max_context_examples_size) is ignored, but total still respected
            if total_tokens + candidate["token_count"] <= max_context_examples_size:
                context_examples.append(candidate)
                total_tokens += candidate["token_count"]
                seen_ids.add(candidate["conversation_id"])

    # Pass 3: Final push to ensure num_desired_contexts examples, by picking smallest remaining
    # if necessary, even if it slightly exceeds the max_context_examples_size for the sum.
    while len(context_examples) < num_desired_contexts and len(seen_ids) < len(sorted_contexts):
        smallest_unseen_candidate = None
        # Find the smallest available candidate not already added
        for candidate in sorted_contexts: # Iterate original sorted_contexts for original sorting priority for ties
            if candidate["conversation_id"] not in seen_ids:
                if smallest_unseen_candidate is None or candidate["token_count"] < smallest_unseen_candidate["token_count"]:
                    smallest_unseen_candidate = candidate

        if smallest_unseen_candidate:
            context_examples.append(smallest_unseen_candidate)
            total_tokens += smallest_unseen_candidate["token_count"] # Update total, may exceed budget
            seen_ids.add(smallest_unseen_candidate["conversation_id"])
        else:
            break # No more candidates to add, cannot reach num_desired_contexts

    context_examples = context_examples[:num_desired_contexts] # Ensure exactly 5 if more were added (unlikely)


    context_ids = {c["conversation_id"] for c in context_examples}
    # final_records = [r for r in records if r["conversation_id"] not in context_ids]
    # temporary to not remove the context records
    final_records = records

    total_relations = total_supports + total_attacks
    if total_relations > 0:
        support_pct = 100 * total_supports / total_relations
        attack_pct = 100 * total_attacks / total_relations

    stats = {
        "total_conversations": total_conversations,
        "avg_conv_length": total_conv_tokens / total_conversations,
        "total_argument_units": total_argument_units,
        "avg_argument_units_per_conversation": total_argument_units / total_conversations,
        "total_supports": total_supports,
        "avg_supports_per_conversation": total_supports / total_conversations,
        "total_attacks": total_attacks,
        "avg_attacks_per_conversation": total_attacks / total_conversations,
        "support_percentage": support_pct,
        "attack_percentage": attack_pct,
        "total_dropped_units": total_dropped_units,
    }


    for r in final_records + context_examples:
        r.pop("relation_counts", None)
        r.pop("diversity_score", None)
        r.pop("weighted_relations", None)
        r.pop("token_count", None)

    df = pd.DataFrame(final_records)
    return df, context_examples, stats


In [ ]:
if __name__ == "__main__":

    input_dirs = "Raw JSON Argument Maps (IAT)"
    repurposed_json_folder = "/content/content/repurposed_maps"
    output_dir_for_corpus = "final"

    os.makedirs(repurposed_json_folder, exist_ok=True)
    os.makedirs(output_dir_for_corpus, exist_ok=True)

    # For collecting all dataset statistics
    all_stats = []

    for input_dir in os.listdir(input_dirs):
        input_dir = os.path.join(input_dirs, input_dir)

        # repurpose the argument maps

        develop_argument_map_from_corpus(input_dir, repurposed_json_folder)

        # add conversation text, do initial cleaning, filter out short graphs, develop the final csv and context examples.

        folder_name = str(os.path.basename(input_dir))

        # develop the new data files, and context examples
        df, context_examples, stats = develop_data_files(input_dir, repurposed_json_folder)
        # attach dataset name to stats and record it
        stats["dataset_name"] = folder_name
        all_stats.append(stats)

        # save the data files
        final_data = os.path.join(output_dir_for_corpus, "final_data")
        os.makedirs(final_data, exist_ok=True)
        df.to_csv(os.path.join(final_data, f"{folder_name}.csv"), index=False)

        # save the context examples
        complex_examples_path = os.path.join(output_dir_for_corpus, "context_examples")
        os.makedirs(complex_examples_path, exist_ok=True)
        context_path = os.path.join(complex_examples_path, f"{folder_name}_context_examples.json")
        with open(context_path,  "w") as f:
            json.dump(context_examples, f, indent=2)

        for i, ctx in enumerate(context_examples):

          print(f"Context example {i+1} (ID: {ctx['conversation_id']}) has {count_tokens(ctx['conversation_text'])} tokens.")


        print(f"Saved {len(df)} rows to CSV and {len(context_examples)} context examples to JSON.")

    # Convert all collected stats into a summary dataframe
    stats_df = pd.DataFrame(all_stats)
    # Save final stats summary to CSV
    stats_path = os.path.join(output_dir_for_corpus, "corpus_statistics.csv")
    stats_df.to_csv(stats_path, index=False)

New directory 'RIP1' created at /content/content/repurposed_maps/RIP1

[Repaired nodes] Conversation 29802
  - Node 8 (score=0.632): 'Female 1: it was Chris' -> 'is this'

[Dropped nodes] Conversation 29802
  - Node 2: Female 1: he changed his will on the 3rd of December
  - Node 3: Female 1: there's no examples of a U
  - Node 4: Female 1: it matches Chris' one
  - Node 5: Female 1: it could be a bad U
  - Node 6: Female 1: there are loose ends
  - Node 7: Female 1: we could say it
  - Node 9: Female 2: that was on the 3rd of December
  - Node 10: Female 1: The code repeats

[Repaired nodes] Conversation 29917
  - Node 1 (score=0.973): 'Female 3: we're assuming we trust Cherie's alibi' -> 'we're assuming we trust Cherie's alibi'
  - Node 9 (score=0.979): 'Female 3: there's no Mercedes Benz' -> 'there's no Mercedes Benz'

[Dropped nodes] Conversation 29917
  - Node 3: Female 1: I've not seen any evidence that would confirm that
  - Node 4: Female 1: I don't think it's Carmen
  - Node 5

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Streaming output truncated to the last 5000 lines.
  - Node 9: Steve Barclay : the construction industry is a key part of that
  - Node 10: Steve Barclay : We're committed to funding that
  - Node 11: Steve Barclay : we also need to make the planning system more friendly to delivering on that

[Repaired nodes] Conversation 23860
  - Node 10 (score=0.993): 'Hugh Osmond :  it is almost exclusively in the younger part of the market, low-paid jobs' -> 'it is almost exclusively in the younger part of the market, low-paid jobs'

[Dropped nodes] Conversation 23860
  - Node 2: Hugh Osmond : This means through COVID and through the pandemic and coming out of it it appears that the rich get richer and the poor get poorer, as a huge generalisation
  - Node 3: John Bercow : People want a protective state and a Labour Party
  - Node 4: John Bercow : there is a market for a left-of-centre party that can show how it will tackle the levelling up agenda, how it will help the left-behind, the left-out, 

In [ ]:
!zip repurposed_maps.zip -r /content/content/repurposed_maps

In [ ]:
!zip final.zip -r /content/final

## Prepare Data for Fine-tuning


- For Unified model
- For AUE model
- For RTC model

In [ ]:
# Split Qt30 into training and test set

qt_complete = pd.read_csv("qt30_complete.csv")

qt_complete.head()

,conversation_id,conversation_text,argument_objects,dropped_units
0,20305,Fiona Bruce [0:09:04] Angela. Let's bring you ...,"{""argument_units"": [{""reason"": ""Asserting"", ""i...",0
1,21586,"Fiona Bruce[0:13:26] Just to be clear, the mom...","{""argument_units"": [{""reason"": ""Asserting"", ""i...",0
2,18320,Fiona Bruce[0:43:10] If you have just beenHele...,"{""argument_units"": [{""reason"": ""Asserting"", ""i...",0
3,25557,Steve Barclay[0:14:26] You hear from the perso...,"{""argument_units"": [{""reason"": ""Asserting"", ""i...",0
4,23848,"Hugh Osmond[0:07:50] Well, I think when people...","{""argument_units"": [{""reason"": ""Asserting"", ""i...",0


In [ ]:
# calculate the number of attacks in the conversation

qt_complete["attack_count"] =


In [ ]:
from sklearn.model_selection import train_test_split


training_set, testing_set = train_test_split(
    qt_complete,
    test_size=0.2,
    random_state=42,
)

training_set.reset_index(drop=True, inplace=True)
testing_set.reset_index(drop=True, inplace=True)

training_set.describe()

,conversation_id,dropped_units
count,669.000000,669.000000
mean,21579.605381,0.005979
std,2279.944592,0.077151
min,17918.000000,0.000000
25%,19324.000000,0.000000
50%,21391.000000,0.000000
75%,23729.000000,0.000000
max,26087.000000,1.000000


In [ ]:
testing_set.describe()

,conversation_id,dropped_units
count,168.000000,168.000000
mean,21807.636905,0.011905
std,2151.357308,0.108782
min,17921.000000,0.000000
25%,19863.750000,0.000000
50%,21523.000000,0.000000
75%,23624.750000,0.000000
max,26068.000000,1.000000


In [ ]:
training_set.to_csv("QT30_training.csv")
testing_set.to_csv("QT30_test.csv")

In [ ]:
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd, json, ast, os
import random

In [ ]:
MAX_SEQ_LEN = 4096
MAX_CONTEXT_LEN = int(MAX_SEQ_LEN * 0.75)
tokenizer = BertTokenizer.from_pretrained("bert-large-cased")

def get_context_span(convo_text, src_text, tgt_text, tokenizer):
    s_idx = convo_text.find(src_text)
    t_idx = convo_text.find(tgt_text)
    if s_idx == -1 or t_idx == -1:
        return ''
    start = min(s_idx, t_idx)
    end = max(s_idx + len(src_text), t_idx + len(tgt_text))
    context = convo_text[start:end]
    tokens = tokenizer.tokenize(context)
    if len(tokens) > MAX_SEQ_LEN:
        tokens = tokens[:MAX_CONTEXT_LEN]
        context = tokenizer.decode(tokenizer.convert_tokens_to_ids(tokens))
    return context

def prepare_relation_and_extraction_data(
    input_csv_path: str,
    output_dir: str,
    test_size: float = 0.20,
    random_seed: int = 42
):
    df = pd.read_csv(input_csv_path)
    df['argument_objects'] = df['argument_objects'].apply(lambda x: ast.literal_eval(x))

    extraction_entries = {}
    relation_entries = []
    attack_counts = {}

    # iterate over each conversation

    for _, row in df.iterrows():
        convo_id = str(row['conversation_id'])
        convo_text = row['conversation_text']
        args_data = row['argument_objects']
        units = args_data.get('argument_units', [])
        supports = args_data.get('support_relationships', [])
        attacks = args_data.get('attack_relationships', [])

        # Extraction data
        clauses = [u['text'] for u in units if u.get('text', '').strip()]
        extraction_entries[convo_id] = {
            'id': convo_id,
            'text': convo_text,
            'units': clauses
        }

        # Record attack count for stratification
        attack_counts[convo_id] = len(attacks)

        # Build id->text
        id2text = {u['id']: u['text'] for u in units}
        all_ids = list(id2text.keys())

        # Support
        for rel in supports:
            src, tgt = rel["source_id"], rel["target_id"]
            relation_entries.append({
                'conv_id': convo_id,
                'arg1': id2text[src],
                'arg2': id2text[tgt],
                'context': get_context_span(convo_text, id2text[src], id2text[tgt], tokenizer),
                'label': 'support'
            })

        # Attack
        for rel in attacks:
            src, tgt = rel["source_id"], rel["target_id"]
            relation_entries.append({
                'conv_id': convo_id,
                'arg1': id2text[src],
                'arg2': id2text[tgt],
                'context': get_context_span(convo_text, id2text[src], id2text[tgt], tokenizer),
                'label': 'attack'
            })

        # Hard Negatives (no known relation between args)
        hard_negatives = []
        for src in all_ids:
            for tgt in all_ids:
                if src == tgt:
                    continue
                if not any(src == r["source_id"] and tgt == r["target_id"] for r in supports + attacks):
                    hard_negatives.append({
                        'conv_id': convo_id,
                        'arg1': id2text[src],
                        'arg2': id2text[tgt],
                        'context': get_context_span(convo_text, id2text[src], id2text[tgt], tokenizer),
                        'label': 'non',  # hard negative
                        'type': 'hard'
                    })

        # Soft Negatives (arg vs. unrelated sentence)
        for unit in units:
            unit_text = unit['text']
            unit_pos = convo_text.find(unit_text)
            before = convo_text[:unit_pos]
            after = convo_text[unit_pos + len(unit_text):]
            candidate = ""

            for seg in before.split('\n')[::-1] + after.split('\n'):
                seg = seg.strip()
                if seg and seg not in id2text.values() and len(seg) > 20:
                    candidate = seg
                    break

            if candidate:
                relation_entries.append({
                    'conv_id': convo_id,
                    'arg1': unit_text,
                    'arg2': candidate,
                    'context': get_context_span(convo_text, unit_text, candidate, tokenizer),
                    'label': 'non',  # soft negative
                    'type': 'soft'
                })

        # Add hard negatives (limit to same count as positive for balance)
        num_pos = len(supports) + len(attacks)
        relation_entries.extend(hard_negatives[:num_pos])

    # 1. Separate by label
    positive_relations = [r for r in relation_entries if r["label"] in ("support", "attack")]
    non_relations = [r for r in relation_entries if r["label"] == "non"]

    # 2. Further split non-relations
    soft_non = [r for r in non_relations if r.get("type") == "soft"]
    hard_non = [r for r in non_relations if r.get("type") == "hard"]

    # 3. Compute counts
    num_pos = int(1.25 * len(positive_relations))
    min_soft = int(0.2 * num_pos)
    remaining_needed = num_pos - min_soft

    # 4. Sample with fallback protection
    random.seed(random_seed)
    soft_sample = soft_non if len(soft_non) <= min_soft else random.sample(soft_non, min_soft)
    hard_sample = hard_non if len(hard_non) <= remaining_needed else random.sample(hard_non, remaining_needed)

    # 5. Combine and shuffle
    balanced_non = soft_sample + hard_sample
    relation_entries = positive_relations + balanced_non
    random.shuffle(relation_entries)

    convo_ids = list(extraction_entries.keys())

    def bucket_attack_count(n):
      # no attacks
      if n == 0:
          return 0
      # single attack
      elif n == 1:
          return 1
      # small attack space
      elif n <= 3:
          return 2
      # large attack space
      else:
          return 3

    y = [bucket_attack_count(attack_counts[cid]) for cid in convo_ids]

    # y = [attack_counts[cid] for cid in convo_ids]  # use attack count for stratification

    train_ids, test_ids = train_test_split(
        convo_ids,
        test_size=test_size,
        stratify=y,
        random_state=random_seed
    )
    train_ids, test_ids = set(train_ids), set(test_ids)

    # Filter extraction entries
    train_arg = [extraction_entries[cid] for cid in train_ids]
    test_arg = [extraction_entries[cid] for cid in test_ids]

    # Filter relation entries
    train_rel = [r for r in relation_entries if r['conv_id'] in train_ids]
    test_rel = [r for r in relation_entries if r['conv_id'] in test_ids]
    # Save
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)

    def save(name, items):
        with open(out / name, "w") as f:
            for item in items:
                f.write(json.dumps(item) + "\n")

    save("argument_extraction_train.jsonl", train_arg)
    save("argument_extraction_test.jsonl", test_arg)
    save("relation_balanced_train.jsonl", train_rel)
    save("relation_balanced_test.jsonl", test_rel)


    print(f"Saved splits to: {output_dir}")
    print(f"Train: {len(train_rel)} relations, {len(train_arg)} conversations")
    print(f"Test: {len(test_rel)} relations, {len(test_arg)} conversations")
    print(f"Train: Number of arguments: {sum(len(e['units']) for e in train_arg)}")
    print(f"Test: Number of arguments: {sum(len(e['units']) for e in test_arg)}")
    print(f"Number of Support labels: {len([r for r in relation_entries if r['label'] == 'support'])}, "
          f"Attack labels: {len([r for r in relation_entries if r['label'] == 'attack'])}, "
          f"No relation labels: {len([r for r in relation_entries if r['label'] == 'non'])}")
    print(f"Number of Support labels in train: {len([r for r in train_rel if r['label'] == 'support'])}, "
      f"Attack labels in train: {len([r for r in train_rel if r['label'] == 'attack'])}, "
      f"No relation labels in train: {len([r for r in train_rel if r['label'] == 'non'])}")

    print(f"Number of Support labels in test: {len([r for r in test_rel if r['label'] == 'support'])}, "
      f"Attack labels in test: {len([r for r in test_rel if r['label'] == 'attack'])}, "
      f"No relation labels in test: {len([r for r in test_rel if r['label'] == 'non'])}")


In [ ]:
input_folder = "/content/ArgMining2025/Data/version4/final/final_data/QT30"
output_base = "/content/fine_tuning"

for file in os.listdir(input_folder):
    if not file.endswith(".csv"):
        continue
    input_csv = os.path.join(input_folder, file)
    dataset_name = file.replace(".csv", "")
    output_dir = os.path.join(output_base, dataset_name)
    prepare_relation_and_extraction_data(input_csv, output_dir)


Saved splits to: /content/fine_tuning/qt30
Train: 9060 relations, 558 conversations
Test: 2271 relations, 140 conversations
Train: Number of arguments: 5959
Test: Number of arguments: 1490
Number of Support labels: 4330, Attack labels: 706, No relation labels: 6295
Number of Support labels in train: 3472, Attack labels in train: 565, No relation labels in train: 5023
Number of Support labels in test: 858, Attack labels in test: 141, No relation labels in test: 1272


In [ ]:
import json
import pandas as pd

In [ ]:
# split qt30.csv file based on the training conv id from training.json

qt30_csv = "/content/ArgMining2025/Data/final/train_files/qt30.csv"
qt30_df = pd.read_csv(qt30_csv)
print(len(qt30_df))
with open("/content/ArgMining2025/Data/final/train_files/fine_tuning/qt30/argument_extraction_train.jsonl") as f:
  data = [json.loads(line) for line in f]

print(len(data))
# conv id in training
conv_ids = [int(r["id"]) for r in data]
print(len(conv_ids))
# split qt30 df into train and test
qt30_train = qt30_df[qt30_df["conversation_id"].isin(conv_ids)]
qt30_test = qt30_df[~qt30_df["conversation_id"].isin(conv_ids)]
print(len(qt30_train), len(qt30_test))
# save
qt30_train.to_csv("/content/qt30_train.csv", index=False)
qt30_test.to_csv("/content/qt30_test.csv", index=False)

698
558
558
558 140


In [ ]:
qt30_train = pd.read_csv("/content/qt30_train.csv")
qt30_test = pd.read_csv("/content/qt30_test.csv")

# calculate average conversation token length

qt30_train["token_count"] = qt30_train["conversation_text"].apply(count_tokens)
qt30_test["token_count"] = qt30_test["conversation_text"].apply(count_tokens)

print(qt30_train["token_count"].mean())
print(qt30_test["token_count"].mean())

Token indices sequence length is longer than the specified maximum sequence length for this model (528 > 512). Running this sequence through the model will result in indexing errors


299.15232974910396
304.2857142857143


In [ ]:
!zip final.zip -r /content/fine_tuning/qt30

  adding: content/fine_tuning/qt30/ (stored 0%)
  adding: content/fine_tuning/qt30/argument_extraction_test.jsonl (deflated 73%)
  adding: content/fine_tuning/qt30/argument_extraction_train.jsonl (deflated 73%)
  adding: content/fine_tuning/qt30/relation_balanced_train.jsonl (deflated 74%)
  adding: content/fine_tuning/qt30/relation_balanced_test.jsonl (deflated 77%)
